# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Encode categorical features using ordinal and one-hot strategies. 
- Handle unknown, infrequent, and redundant categories in practical datasets. 
- Integrate categorical encoders into ColumnTransformer workflows. 


## **1. Basic Categorical Encoders**

### **1.1. Ordinal Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_01.jpg?v=1783907309" width="250">



>* Convert ordered categories into integer codes
>* Preserve meaningful ranking for numeric models

>* Numbers can imply meaningful category distances
>* Use one-hot encoding for unordered categories

>* Define category order deliberately and consistently
>* Treat encoded numbers as labels, not measurements



In [ ]:
#@title Python Code - Ordinal Encoding

# Demonstrate ordinal encoding with ordered categories.
# Use only built-in Python data structures.
# Keep output short for beginner learners.

# Create tiny training and future datasets.
train_sizes = ["small", "medium", "large", "medium"]
future_sizes = ["large", "small", "extra large"]
ordered_sizes = ["small", "medium", "large"]

# Build a deliberate category-to-number mapping.
size_to_number = {size: index for index, size in enumerate(ordered_sizes)}
unknown_value = -1
encoded_train = [size_to_number.get(size, unknown_value) for size in train_sizes]

# Encode future values using the same mapping.
encoded_future = [size_to_number.get(size, unknown_value) for size in future_sizes]
train_pairs = list(zip(train_sizes, encoded_train))
future_pairs = list(zip(future_sizes, encoded_future))

# Show compact results without printing large objects.
print("Ordinal order:", ordered_sizes)
print("Training encoding:", train_pairs)
print("Future encoding:", future_pairs)
print("Unknown categories use:", unknown_value)

# Compare with a misleading unordered example.
colors = ["red", "blue", "green"]
color_codes = {color: index for index, color in enumerate(colors)}
print("Unordered color codes:", color_codes)
print("Color numbers are labels, not rankings.")



### **1.2. One Hot Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_02.jpg?v=1783907316" width="250">



>* Creates binary columns for unordered categories
>* Avoids false numeric ranking or distance

>* Preserves categories without false numeric order
>* Lets models learn category-specific outcome patterns

>* High-cardinality features create many columns
>* Redundant indicators can affect some models



In [ ]:
#@title Python Code - One Hot Encoding

# Demonstrate one hot encoding basics.
# Use tiny categorical housing examples.
# Avoid artificial numeric category order.

import pandas as pd

# Create a small categorical dataset.
homes = pd.DataFrame(
    {
        "neighborhood": ["North", "South", "East", "North"],
        "fuel_type": ["Gas", "Electric", "Gas", "Oil"],
    }
)

# Encode categories into indicator columns.
encoded = pd.get_dummies(
    homes,
    columns=["neighborhood", "fuel_type"],
    dtype=int,
)

# Validate the expected row count.
assert encoded.shape[0] == homes.shape[0]

# Show the original compact data.
print("Original categorical data:")
print(homes.to_string(index=False))

# Show the one hot encoded result.
print("\nOne hot encoded columns:")
print(encoded.to_string(index=False))

# Summarize the column expansion.
print(f"\nColumns before: {homes.shape[1]}, after: {encoded.shape[1]}")



### **1.3. Pandas Categories**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_03.jpg?v=1783907311" width="250">



>* Mark repeated labels as defined categories
>* Prepare categories for machine learning encoding

>* Categories store repeated labels efficiently
>* Internal codes are not model encodings

>* Clean categories reveal inconsistent or unexpected values
>* Defined categories support one-hot or ordinal encoding



In [ ]:
#@title Python Code - Pandas Categories

# Pandas categories clarify repeated text labels.
# Internal codes are storage details only.
# Ordered categories can preserve meaningful progression.

# Import pandas for categorical data handling.
import pandas as pd

# Create a tiny dataset with repeated labels.
data = pd.DataFrame(
    {
        "size": ["small", "medium", "large", "medium"],
        "color": ["red", "blue", "red", "green"],
    }
)

# Define an ordered category for product size.
size_type = pd.CategoricalDtype(
    categories=["small", "medium", "large"],
    ordered=True,
)

# Convert size into an ordered categorical column.
data["size"] = data["size"].astype(size_type)

# Convert color into an unordered categorical column.
data["color"] = data["color"].astype("category")

# Show compact category metadata, not full data.
print("Size categories:", list(data["size"].cat.categories))
print("Size ordered:", data["size"].cat.ordered)
print("Color categories:", list(data["color"].cat.categories))

# Show internal codes for storage awareness.
print("Size internal codes:", data["size"].cat.codes.tolist())
print("Color internal codes:", data["color"].cat.codes.tolist())

# Compare memory use before and after conversion.
raw = pd.Series(["medium", "small", "large", "medium"])
cat = raw.astype(size_type)

# Print a small memory comparison summary.
print("Raw bytes:", int(raw.memory_usage(deep=True)))
print("Category bytes:", int(cat.memory_usage(deep=True)))
print("Use encoders later for modeling numbers.")



## **2. Category Handling**

### **2.1. Unknown Category Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_01.jpg?v=1783907325" width="250">



>* New categories can appear after training
>* Plan encoder behavior for unfamiliar values

>* One-hot unknowns can use all-zero indicators
>* Ordinal unknowns may imply false ordering

>* Fit encoders only on training categories
>* Monitor unknowns to trigger reliable retraining



In [ ]:
#@title Python Code - Unknown Category Handling

# Demonstrate unknown category handling.
# Use tiny categorical training data.
# Compare strict and safe encodings.

import pandas as pd

# Create training categories seen during fitting.
train = pd.Series(["North", "South", "East", "West"])

# Create future categories including unknown values.
future = pd.Series(["North", "Central", "West", "Island"])

# Learn categories from training data only.
known_categories = sorted(train.unique().tolist())

# Build one-hot columns for known categories.
one_hot = pd.get_dummies(future)
one_hot = one_hot.reindex(columns=known_categories, fill_value=0)

# Build ordinal mapping with safe unknown code.
ordinal_map = {name: index for index, name in enumerate(known_categories)}
ordinal_safe = future.map(ordinal_map).fillna(-1).astype(int)

# Count unknown categories in future data.
unknown_mask = ~future.isin(known_categories)
unknown_count = int(unknown_mask.sum())

# Show compact teaching results.
print("Known categories:", known_categories)
print("Future values:", future.tolist())
print("Unknown count:", unknown_count)
print("One-hot unknown rows become all zeros:")
print(one_hot.loc[unknown_mask].to_string(index=False))
print("Safe ordinal codes, using -1 for unknown:", ordinal_safe.tolist())



### **2.2. Infrequent Category Grouping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_02.jpg?v=1783907323" width="250">



>* Rare categories create sparse, noisy encodings
>* Group them into an “other” category

>* Group rare categories for more stable models
>* Reduce one-hot features and overfitting risk

>* Choose thresholds based on data context
>* Group using training data to avoid leakage



In [ ]:
#@title Python Code - Infrequent Category Grouping

# Group rare categories before encoding.
# This example uses training counts.
# Unknown values stay separate later.

import pandas as pd

# Create a tiny training dataset.
train = pd.DataFrame({
    "brand": ["Acme", "Acme", "Acme", "Bravo",
              "Bravo", "Coda", "Delta", "Echo"],
    "price_usd": [12, 15, 11, 20, 22, 9, 8, 7]
})

# Create new data with unknown brands.
new_data = pd.DataFrame({
    "brand": ["Acme", "Coda", "Foxtrot", "Echo"],
    "price_usd": [13, 10, 18, 6]
})

# Count categories using training data only.
counts = train["brand"].value_counts()

# Keep categories appearing at least twice.
common = set(counts[counts >= 2].index)

# Group rare training categories as infrequent.
train["brand_grouped"] = train["brand"].where(
    train["brand"].isin(common), "infrequent"
)

# Mark unseen categories as unknown.
new_data["brand_grouped"] = new_data["brand"].where(
    new_data["brand"].isin(common), "unknown_or_infrequent"
)

# One-hot encode the grouped training feature.
encoded_train = pd.get_dummies(
    train["brand_grouped"], prefix="brand"
)

# Align new encoded columns safely.
encoded_new = pd.get_dummies(
    new_data["brand_grouped"], prefix="brand"
)

# Match training columns for modeling workflows.
encoded_new = encoded_new.reindex(
    columns=encoded_train.columns, fill_value=0
)

# Summarize the grouping decision.
print("Training category counts:", counts.to_dict())
print("Common categories kept:", sorted(common))
print("Grouped training labels:", train["brand_grouped"].tolist())
print("Encoded training columns:", encoded_train.columns.tolist())
print("New labels after grouping:", new_data["brand_grouped"].tolist())
print("Aligned new encoded shape:", encoded_new.shape)



### **2.3. Drop Options**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_03.jpg?v=1783907327" width="250">



>* Drop one category to avoid redundancy
>* Use it as the comparison baseline

>* Choose dropped categories for meaningful comparisons
>* Baselines shape stakeholder interpretation

>* Coordinate drops with unknown and rare categories
>* Preserve interpretability, auditability, and fair explanations



In [ ]:
#@title Python Code - Drop Options

# Drop options create reference categories.
# Unknown categories need careful interpretation.
# Tiny examples reveal encoding behavior.

import pandas as pd
import numpy as np

# Create training categories and prediction categories.
train = pd.Series(["cash", "card", "transfer", "cash"])
new = pd.Series(["cash", "card", "crypto"])

# Choose a meaningful baseline category.
categories = ["cash", "card", "transfer"]
dropped_category = "cash"

# Encode known categories except the dropped baseline.
kept_categories = [cat for cat in categories if cat != dropped_category]
encoded = pd.get_dummies(new).reindex(columns=kept_categories, fill_value=0)

# Mark values not seen during fitting.
unknown_mask = ~new.isin(categories)
unknown_count = int(unknown_mask.sum())

# Show compact teaching output.
print("Dropped baseline category:", dropped_category)
print("Encoded columns kept:", list(encoded.columns))
print("New values:", list(new))
print("Encoded rows:", encoded.astype(int).values.tolist())
print("Unknown category count:", unknown_count)
print("All-zero rows can mean baseline or unknown.")



## **3. Encoder Pipelines**

### **3.1. Target Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_01.jpg?v=1783907331" width="250">



>* Replace categories with target-based numeric summaries
>* Useful for high-cardinality pipeline transformations

>* Target encoding can leak target information
>* Use smoothing and pipelines for safer encoding

>* Apply encoders to appropriate feature groups
>* Reuse fitted mappings for consistent production preprocessing



In [ ]:
#@title Python Code - Target Encoding

# Target encoding summarizes categories using outcomes.
# Pipelines keep preprocessing repeatable and safe.
# This example avoids external machine learning libraries.

import pandas as pd
import numpy as np

# Create a tiny housing style dataset.
data = pd.DataFrame({
    "neighborhood": ["A", "A", "B", "B", "C", "D", "D", "E"],
    "home_size_sqft": [900, 1100, 850, 950, 1200, 700, 760, 1300],
    "price_k": [310, 330, 250, 270, 360, 210, 220, 390],
})

# Split rows before learning category summaries.
train = data.iloc[:6].copy()
test = data.iloc[6:].copy()

# Compute the global target average.
global_mean = train["price_k"].mean()
category_stats = train.groupby("neighborhood")["price_k"].agg(["mean", "count"])

# Smooth rare categories toward the global mean.
smoothing_strength = 2
category_stats["encoded"] = (
    category_stats["mean"] * category_stats["count"] + global_mean * smoothing_strength
) / (category_stats["count"] + smoothing_strength)

# Apply learned mappings to train and test data.
train["neighborhood_target_encoded"] = train["neighborhood"].map(category_stats["encoded"])
test["neighborhood_target_encoded"] = test["neighborhood"].map(category_stats["encoded"])

# Unknown categories safely receive the global mean.
test["neighborhood_target_encoded"] = test["neighborhood_target_encoded"].fillna(global_mean)

# Mimic a ColumnTransformer by combining selected outputs.
train_features = pd.concat([
    train[["home_size_sqft"]].reset_index(drop=True),
    train[["neighborhood_target_encoded"]].reset_index(drop=True),
], axis=1)

test_features = pd.concat([
    test[["home_size_sqft"]].reset_index(drop=True),
    test[["neighborhood_target_encoded"]].reset_index(drop=True),
], axis=1)

# Validate the transformed feature matrices.
assert train_features.shape == (6, 2)
assert test_features.shape == (2, 2)

# Show compact results for learners.
print("Global mean price:", round(global_mean, 1))
print("Learned target encodings:")
print(category_stats[["encoded"]].round(1).to_string())
print("Test features after pipeline-style transformation:")
print(test_features.round(1).to_string(index=False))



### **3.2. Sparse Encoder Output**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_02.jpg?v=1783907329" width="250">



>* One-hot encoding creates many mostly-zero columns
>* Sparse output saves memory and computation

>* Sparse output saves memory for compatible models
>* Check downstream steps before choosing sparsity

>* ColumnTransformer combines sparse categorical and numeric features
>* Ensure downstream steps support the output format



### **3.3. ColumnTransformer Workflows**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_03.jpg?v=1783907335" width="250">



>* Apply tailored preprocessing to each column type
>* Combine transformed features into one model matrix

>* Keep category handling consistent across datasets
>* Manage unknown and rare categories reliably

>* Compare encoders within stable preprocessing workflows
>* Embed assumptions for governance and maintenance



# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>


In this lecture, you learned to:
- Encode categorical features using ordinal and one-hot strategies. 
- Handle unknown, infrequent, and redundant categories in practical datasets. 
- Integrate categorical encoders into ColumnTransformer workflows. 

In the next Lecture (Lecture B), we will go over 'Imputation Targets'